# 01 优化框架、瓶颈定位和 Benchmark 设计

目标：把部署优化从“背技术名词”变成“根据指标定位瓶颈并选择手段”。重点练 TTFT、TPOT、吞吐、显存、batch、KV cache、量化、编译和可观测性。


## 1. 指标先行

部署优化不要一上来就量化或换框架。先拆指标：

- TTFT: time to first token，用户看到第一个 token 的时间。
- TPOT: time per output token，decode 阶段平均每 token 时间。
- E2E latency: 完整响应时间。
- Throughput: 每秒输出 token 数或请求数。
- Concurrency: 同时在系统内处理的请求数。
- GPU utilization / memory / KV cache usage: 资源是否真的被用上。
- Error rate / OOM / timeout: 稳定性指标。


In [ ]:
optimization_methods = [
    {
        "symptom": "权重显存太大，模型加载 OOM",
        "methods": "FP16/BF16、INT8/INT4、bitsandbytes、GPTQ/AWQ、GGUF、TensorRT-LLM FP8/INT4",
        "tradeoff": "质量可能下降；速度不一定提升；依赖硬件 kernel 和格式支持",
    },
    {
        "symptom": "长上下文或高并发导致 KV cache OOM",
        "methods": "PagedAttention、paged KV cache、GQA/MQA、限制 max_model_len、prefix cache、请求限流",
        "tradeoff": "上下文窗口、并发和成本要权衡",
    },
    {
        "symptom": "TTFT 高",
        "methods": "prefix/prompt cache、chunked prefill、缩短 prompt、RAG context budget、减少排队、预热模型",
        "tradeoff": "缓存命中依赖业务；chunked prefill 可能影响调度策略",
    },
    {
        "symptom": "TPOT 高，生成慢",
        "methods": "FlashAttention/FlashInfer、TensorRT-LLM、vLLM/SGLang、speculative decoding、量化、tensor parallel",
        "tradeoff": "speculative decoding 依赖 draft 模型命中率；多卡有通信开销",
    },
    {
        "symptom": "吞吐低但单请求延迟还行",
        "methods": "continuous batching、dynamic batching、in-flight batching、调大 max_num_seqs、异步队列",
        "tradeoff": "batch 越大，部分请求排队可能越久",
    },
    {
        "symptom": "CPU/API 层瓶颈",
        "methods": "异步 streaming、连接池、tokenizer 进程隔离、FastAPI/Uvicorn worker、网关限流",
        "tradeoff": "要避免把模型瓶颈误判成 API 瓶颈",
    },
    {
        "symptom": "重复请求多",
        "methods": "prefix cache、prompt cache、embedding cache、RAG cache、结果缓存",
        "tradeoff": "缓存一致性、隐私隔离和过期策略要设计清楚",
    },
]

for item in optimization_methods:
    print("=" * 100)
    print("症状:", item["symptom"])
    print("手段:", item["methods"])
    print("代价:", item["tradeoff"])


## 2. 显存估算：权重 + KV cache + 临时激活

粗略估算能帮助你在面试里判断“为什么 7B 模型能加载但并发上不去”。


In [ ]:
def gib(bytes_count):
    return bytes_count / 1024**3


def weight_memory_gib(params_billion, bytes_per_param):
    return gib(params_billion * 1_000_000_000 * bytes_per_param)


def kv_cache_gib(batch_size, seq_len, num_layers, num_kv_heads, head_dim, bytes_per_value=2):
    # key + value 两份缓存。
    bytes_count = batch_size * seq_len * num_layers * 2 * num_kv_heads * head_dim * bytes_per_value
    return gib(bytes_count)


models = [
    {"name": "0.5B", "params_b": 0.5, "layers": 24, "kv_heads": 2, "head_dim": 64},
    {"name": "7B", "params_b": 7, "layers": 32, "kv_heads": 8, "head_dim": 128},
    {"name": "14B", "params_b": 14, "layers": 40, "kv_heads": 8, "head_dim": 128},
]

for model in models:
    print("=" * 100)
    print(model["name"])
    for dtype, bpp in [("fp16/bf16", 2), ("int8", 1), ("int4 rough", 0.5)]:
        print(f"weights {dtype:10s}: {weight_memory_gib(model['params_b'], bpp):6.2f} GiB")
    for batch, seq in [(1, 4096), (8, 4096), (16, 8192)]:
        cache = kv_cache_gib(batch, seq, model["layers"], model["kv_heads"], model["head_dim"])
        print(f"KV batch={batch:<2d} seq={seq:<5d}: {cache:6.2f} GiB")


## 3. 延迟模型：TTFT 和 TPOT 拆开看

总延迟可以粗略拆成：

```text
E2E latency = queue + tokenize + prefill + first_token_sample + decode + network/postprocess
TTFT        = queue + tokenize + prefill + first_token_sample + first_stream_flush
TPOT        = decode_time / output_tokens
```


In [ ]:
def estimate_latency(prompt_tokens, output_tokens, queue_ms=20, tokenize_ms=8, prefill_ms_per_token=0.08, decode_ms_per_token=18, network_ms=15):
    prefill_ms = prompt_tokens * prefill_ms_per_token
    decode_ms = output_tokens * decode_ms_per_token
    ttft = queue_ms + tokenize_ms + prefill_ms + network_ms
    total = ttft + decode_ms
    tpot = decode_ms / max(output_tokens, 1)
    out_tok_per_s = output_tokens / (total / 1000)
    return {
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "TTFT_ms": ttft,
        "TPOT_ms": tpot,
        "latency_ms": total,
        "out_tok_per_s": out_tok_per_s,
    }


for prompt_tokens, output_tokens in [(256, 128), (2048, 128), (8192, 128), (2048, 512)]:
    result = estimate_latency(prompt_tokens, output_tokens)
    print("=" * 100)
    for key, value in result.items():
        if isinstance(value, float):
            print(f"{key:16s}: {value:.2f}")
        else:
            print(f"{key:16s}: {value}")


## 4. Benchmark 设计模板

一次有效压测至少要固定这些变量：

- 模型：名称、参数量、dtype/量化格式、max_model_len。
- 框架：vLLM/TGI/SGLang/TensorRT-LLM 版本，关键启动参数。
- 硬件：GPU 型号、显存、驱动、CUDA、CPU、内存。
- 请求分布：prompt token 分布、output token 分布、并发数、到达率。
- 指标：TTFT P50/P95/P99、TPOT P50/P95、E2E latency、output tok/s、OOM、错误率。
- 约束：质量不下降、结构化输出正确率、成本预算、SLA。


In [ ]:
benchmark_plan = {
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "frameworks": ["vLLM", "TGI", "SGLang"],
    "hardware": "1x NVIDIA L40S 48GB",
    "traffic": {
        "prompt_tokens": [256, 1024, 4096],
        "output_tokens": [128, 512],
        "concurrency": [1, 4, 16, 64],
    },
    "metrics": ["TTFT_p50", "TTFT_p95", "TPOT_p50", "TPOT_p95", "output_tokens_per_sec", "GPU_memory_peak", "error_rate"],
    "acceptance": {
        "TTFT_p95_ms": 1500,
        "TPOT_p95_ms": 60,
        "error_rate_max": 0.01,
        "quality_regression_allowed": False,
    },
}

for key, value in benchmark_plan.items():
    print(key, "=", value)


## 5. 优化决策函数

这个函数不是生产代码，而是训练你把指标和优化手段联系起来。


In [ ]:
def suggest_optimizations(ttft_ms, tpot_ms, gpu_mem_util, gpu_util, error_rate=0.0, repeated_prefix=False):
    suggestions = []

    if error_rate > 0.01:
        suggestions.append("先查错误率/OOM/timeout，优化性能前保证稳定性")
    if gpu_mem_util > 0.9:
        suggestions.append("显存紧张：降低 max_model_len/concurrency，使用量化，检查 KV cache，考虑 paged KV cache")
    if ttft_ms > 1500:
        suggestions.append("TTFT 高：看排队和 prefill，缩短 prompt，启用 prefix cache/chunked prefill，预热模型")
    if tpot_ms > 60:
        suggestions.append("TPOT 高：换高效 serving engine，检查 FlashAttention/FlashInfer/TensorRT-LLM，考虑 speculative decoding 或 tensor parallel")
    if gpu_util < 0.5 and gpu_mem_util < 0.8:
        suggestions.append("GPU 利用率低：提高并发，启用 continuous batching，排查 tokenizer/API 层瓶颈")
    if repeated_prefix:
        suggestions.append("重复系统 prompt 多：重点评估 prefix cache/prompt cache")
    if not suggestions:
        suggestions.append("指标看起来健康，继续做更真实的流量分布和长稳压测")
    return suggestions


cases = [
    {"ttft_ms": 2200, "tpot_ms": 35, "gpu_mem_util": 0.70, "gpu_util": 0.80, "repeated_prefix": True},
    {"ttft_ms": 600, "tpot_ms": 90, "gpu_mem_util": 0.75, "gpu_util": 0.95},
    {"ttft_ms": 900, "tpot_ms": 45, "gpu_mem_util": 0.96, "gpu_util": 0.70, "error_rate": 0.03},
    {"ttft_ms": 500, "tpot_ms": 30, "gpu_mem_util": 0.40, "gpu_util": 0.25},
]

for case in cases:
    print("=" * 100)
    print(case)
    for suggestion in suggest_optimizations(**case):
        print("-", suggestion)


## 6. 框架和优化手段对照

| 技术 | 常见落地位置 | 面试要点 |
| --- | --- | --- |
| PagedAttention / paged KV cache | vLLM、TensorRT-LLM、SGLang 等 | 减少 KV cache 碎片，提高并发利用率 |
| continuous batching | vLLM、SGLang、LMDeploy 等 | 动态把不同请求合进 batch，提高吞吐 |
| dynamic batching | Triton 等通用 serving | 等一小段时间合 batch，适合传统模型，也可用于部分 LLM 后端 |
| FlashAttention | 训练和 prefill/attention kernel | 降低 attention 中间显存和内存访问 |
| FlashInfer | LLM serving kernel | 面向 prefill/decode、paged KV、sampling 等服务场景 |
| bitsandbytes | Transformers/PEFT | 8bit/4bit 加载和 QLoRA，部署时看硬件支持 |
| TensorRT-LLM | NVIDIA 专项优化 | engine 编译、FP8/INT4、in-flight batching、多卡并行 |
| ONNX Runtime / Optimum | 通用模型优化 | 导出、图优化、量化，LLM 支持要看模型结构 |
| torch.compile | PyTorch 编译 | 减少 Python/算子开销，但动态 shape 和首次编译要评估 |


## 7. 面试回答模板

> 我会先用指标定位瓶颈。TTFT 高通常看排队、tokenize 和 prefill，可以缩短 prompt、做 prefix cache、chunked prefill 或优化调度；TPOT 高通常看 decode kernel、KV cache 读取和 batch，可以考虑 vLLM/SGLang/TensorRT-LLM、FlashAttention/FlashInfer、speculative decoding 或多卡并行；显存高要区分权重显存和 KV cache，权重可以降精度/量化，KV cache 要限制上下文和并发、使用 paged KV cache 或 GQA/MQA。任何优化都要通过固定请求分布的 benchmark 验证 TTFT/TPOT/吞吐/质量/错误率，而不是只看单条 demo。
